# Parametric heating — trap frequencies from the loss resonances

For runs of `experiments/Parametric_heating.py` (`*-ParametricHeating.h5`). Modulating the FORT depth at $f_{mod}$ parametrically heats the atom, and loss is resonant at

$$f_{mod} = 2\,f_{trap} \qquad\Longrightarrow\qquad f_{trap} = f_{dip}/2$$

so each dip in the survival curve marks a mode.

- **A** — survival vs modulation frequency, Wilson error bars, multi-dip fit, 0 Hz control line
- **B** — shot-1 / shot-2 count histograms against the threshold: is the atom call clean?
- **C** — loading and survival vs scan repeat: did anything drift during the run?

## Two things this notebook does not take on trust

**1. The atom threshold.** The experiment decided `loaded`/`survived` on the core device with whatever `atom_threshold` was live — but it stored the raw counts too, so the decision is remade here. Set `THRESHOLD` below; `None` uses the run's archived value, `"stored"` uses the kernel's own booleans. The summary reports how many shot decisions the choice flips. Put the threshold in the **valley** of panel B.

**2. How many dips there are.** A single Gaussian is the wrong model for a real trap — an anisotropic FORT has two radial frequencies. The fit is a *sum* of Gaussian dips, added one at a time at the largest residual and refined by coordinate descent. `N_DIPS = None` keeps adding only while each addition improves $\chi^2$ by >25 (~5σ), which is deliberately strict: a greedy fitter always finds *something*. Set `N_DIPS = 3` to force a count.

Not every feature is a separate mode — a shallow extra dip is often a **harmonic or sub-harmonic** of a real resonance ($2f_{trap}/n$) rather than a new frequency, which is the main reason the addition criterion is strict.

The baseline $S_0$ is **pinned** to the measured no-heating plateau rather than fitted. Letting it float lets a flat plateau be re-expressed as a huge $S_0$ minus two broad edge dips — which is exactly what the first version of this fit did (it returned $S_0 = -0.42$ with 85 kHz-wide dips at the scan edges). Widths are capped at span/8 for the same reason.

Linear algebra is hand-rolled Gaussian elimination — no `np.linalg`, no scipy — because the BLAS in `artiq7_claude` segfaults, and a longhand solve also can't silently fail to converge the way `curve_fit` can.

All logic is imported from `analyze_parametric_heating.py`; this notebook keeps no copy.

In [ ]:
# ---------------------------------------------------------------------------
# KERNEL CHECK -- run this FIRST.
# matplotlib cannot draw in the `artiq7_claude` env (numpy 1.23.5, broken BLAS:
# its affine transforms call np.dot while rendering, so plotting SEGFAULTS and
# the client reports "The Kernel crashed"). This analysis needs only
# numpy / h5py / matplotlib, NOT artiq -- so run it on plain Python 3.11:
#     VS Code -> kernel picker (top right) -> Python 3.11
# The probe runs np.dot in a SUBPROCESS, so a broken BLAS kills the child
# process instead of this kernel.
# ---------------------------------------------------------------------------
import subprocess, sys

_blas = subprocess.run(
    [sys.executable, "-c",
     "import numpy as np; np.dot(np.ones((8, 8)), np.ones((8, 8)))"],
    capture_output=True)
if _blas.returncode != 0:
    print("!" * 74)
    print("!! BROKEN BLAS in this kernel -- plotting WILL crash it.")
    print("!!   kernel : {}".format(sys.executable))
    print("!!   fix    : switch the kernel to Python 3.11, then re-run.")
    print("!" * 74)
else:
    print("BLAS ok -- plotting is safe in this kernel.")
    print("kernel:", sys.executable)

# Backend by what is importable -- never a bare `%matplotlib widget` (it needs
# ipympl; without it the kernel falls back to Qt, where plt.show() blocks).
try:
    import ipympl  # noqa: F401
    get_ipython().run_line_magic("matplotlib", "widget")
except ImportError:
    get_ipython().run_line_magic("matplotlib", "inline")

import os, importlib
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

sys.path.insert(0, os.getcwd())
import analyze_parametric_heating as A
importlib.reload(A)          # picks up edits to the .py without restarting
from analyze_parametric_heating import (find_run, find_runs, load_run,
                                        survival_curve, fit_dips, wilson,
                                        sum_resonance_check,
                                        summarize, analyze)

print("matplotlib backend:", matplotlib.get_backend())
print("runs on disk:", [A._rid_of(p) for p in find_runs()])

## Settings

In [ ]:
RID       = None   # None = latest run; or e.g. 2678
THRESHOLD = 10     # counts on (shot - background). None = the run's archived
                   # value; "stored" = the kernel's own booleans.
N_DIPS    = None   # None = add dips only while each is worth >5 sigma.
                   # Set an integer to force a count.

## One run

In [ ]:
fig = analyze(find_run(RID), threshold=THRESHOLD, n_dips=N_DIPS)

## Just the numbers

In [ ]:
res = summarize(find_run(RID), threshold=THRESHOLD, n_dips=N_DIPS, verbose=False)
fit = res["fit"]
print("RID %s | %d shots | %d freqs | threshold %.0f | depth %.3f | t_mod %.2f ms"
      % (res["rid"], res["n_shots"], res["n_freqs"], res["threshold"],
         res["mod_depth"], 1e3 * res["t_mod"]))
print("0 Hz control survival : %.3f    fitted baseline : %.3f"
      % (res["baseline_0hz"], fit["baseline"] if fit else float("nan")))
print()
if fit is None:
    print("no fit (too few frequency points)")
else:
    for i, d in enumerate(sorted(fit["dips"], key=lambda x: x["fc"]), 1):
        flag = "  SATURATED -- centre unreliable" if d["saturated"] else ""
        print("dip %d: f_mod = %7.2f +- %.2f kHz  (FWHM %5.2f, depth %.3f)"
              "  ->  f_trap = %6.2f kHz%s"
              % (i, d["fc"] / 1e3, d["fc_err"] / 1e3, d["fwhm"] / 1e3,
                 d["amplitude"], d["f_trap"] / 1e3, flag))
    print()
    print("chi2/dof = %.2f over %d points" % (fit["chi2_red"], fit["n_points"]))

## Threshold sensitivity

Does the answer depend on where the atom cut is drawn? If the dip centres move by more than their error bars, the threshold is sitting on a peak rather than in the valley — fix that before believing the frequencies.

In [ ]:
print(" thr  loaded   dips: centre [kHz] (f_trap)")
for thr in (8, 10, 12, 14, 16):
    r = summarize(find_run(RID), threshold=thr, n_dips=N_DIPS, verbose=False)
    if r is None or r["fit"] is None:
        print(" %3d   %5.1f%%   (no fit)" % (thr, 100 * r["load_fraction"]))
        continue
    ds = sorted(r["fit"]["dips"], key=lambda x: x["fc"])
    txt = "   ".join("%.2f (%.2f)" % (d["fc"] / 1e3, d["f_trap"] / 1e3)
                     for d in ds)
    print(" %3d   %5.1f%%   %s" % (thr, 100 * r["load_fraction"], txt))

## The survival table

Per-frequency counts, so you can see how much data sits behind each point. Thin points carry wide error bars and pull the fit little.

In [ ]:
data, meta, rid = load_run(find_run(RID), threshold=THRESHOLD)
c = survival_curve(data)

print("  f_actual [kHz]   loaded  survived   survival  68% interval")
for i in np.argsort(c["f"]):
    print("      %9.3f      %4d      %4d     %.3f    [%.3f, %.3f]"
          % (c["f"][i] / 1e3, c["n_loaded"][i], c["n_survived"][i],
             c["p"][i], c["lo"][i], c["hi"][i]))

## Compare runs

Overlay runs taken at different `mod_depth` or `t_mod`. A **real mode** sits at the same frequency and only deepens/widens with more drive. A feature that **moves** with the drive, or that appears only at strong drive, is more likely a harmonic or a power-broadened skirt.

In [ ]:
RIDS = [A._rid_of(p) for p in find_runs()][-3:]    # last 3 runs; edit freely

plt.figure(figsize=(9.5, 5))
for rid in RIDS:
    r = summarize(find_run(rid), threshold=THRESHOLD, n_dips=N_DIPS,
                  verbose=False)
    if r is None:
        continue
    cc = r["curve"]
    m = cc["f"] > 0
    plt.errorbar(cc["f"][m] / 1e3, cc["p"][m],
                 yerr=[np.maximum(0, cc["p"][m] - cc["lo"][m]),
                       np.maximum(0, cc["hi"][m] - cc["p"][m])],
                 fmt="-o", ms=3, lw=1, capsize=2, alpha=0.8,
                 label="RID %d  depth %.3f, %.1f ms"
                       % (rid, r["mod_depth"], 1e3 * r["t_mod"]))
    if r["fit"] is not None:
        for d in r["fit"]["dips"]:
            plt.axvline(d["fc"] / 1e3, ls=":", lw=1, alpha=0.5)
plt.xlabel("modulation frequency played [kHz]")
plt.ylabel("survival (survived / loaded)")
plt.ylim(0, 1.05)
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()

## Reading the result — check before believing it

1. **Panel B** — the two histograms should be clearly bimodal with the threshold in the valley. If it cuts a peak, both `loaded` and `survived` are miscalled. Cross-check with the threshold-sensitivity cell above.
2. **Baseline** — the fitted $S_0$ should agree with the 0 Hz control. If the control is much lower, atoms are being lost for reasons other than modulation (trap lifetime, imaging heating) and every dip sits on a sloping floor.
3. **Panel C** — loading and survival should be flat across repeats; a trend smears the dips.
4. **Saturation** — a dip floored near zero survival has a flat bottom, so its centre is poorly defined and its quoted error is meaningless. Re-measure with weaker drive until the minimum sits near 0.3–0.5. The summary flags this per dip.
5. **Which dips are real modes** — two dips at $2f_x$ and $2f_y$ is the natural reading for an anisotropic trap. A third, shallower feature is more often a harmonic of a real resonance than a third mode; the drive-strength comparison above is how to tell. If you *do* believe three, `sum_resonance_check()` tests the $2f_x$ / $f_x{+}f_y$ / $2f_y$ pattern.
6. `mod_depth` is a fraction of the **DDS amplitude**, not of the trap depth — the AOM is nonlinear. Dip *positions* are unaffected; a heating *rate* needs a photodiode calibration.